In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
feat = pd.read_csv('credit_card_featured.csv')

In [3]:
raw_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
       
payment_ratio_cols = ['PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY']
util_rate_cols = ['UTIL_SEP', 'UTIL_AUG', 'UTIL_JUL', 'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR']
aggregate_cols = ['MONTHS_NO_BALANCE', 'BILL_GROWTH_RATIO', 'UTIL_TREND', 'PAY_TREND', 
       'UTIL_AVG', 'UTIL_MAX', 'UTIL_MIN', 'UTIL_STD']

In [4]:
current_month = feat[feat['SEP_REPAY_STATUS'] <= 0]
y = current_month['default.payment.next.month']
X = current_month[raw_cols + payment_ratio_cols + util_rate_cols + aggregate_cols]


In [5]:
X.columns

Index(['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'SEP_BILL_AMT',
       'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT', 'MAY_BILL_AMT',
       'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT', 'JUL_PAY_AMT',
       'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT', 'PAY_RATIO_SEP',
       'PAY_RATIO_AUG', 'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY',
       'UTIL_SEP', 'UTIL_AUG', 'UTIL_JUL', 'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR',
       'MONTHS_NO_BALANCE', 'BILL_GROWTH_RATIO', 'UTIL_TREND', 'PAY_TREND',
       'UTIL_AVG', 'UTIL_MAX', 'UTIL_MIN', 'UTIL_STD'],
      dtype='str')

In [6]:
X.shape

(23182, 36)

In [7]:
y.mean()

np.float64(0.13834009145026313)

In [8]:
current_month.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT',
       'default.payment.next.month', 'AGE_GROUP', 'EDUCATION_LEVEL',
       'MARRIAGE_DESC', 'LIMIT_BIN', 'PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY', 'UTIL_SEP',
       'UTIL_AUG', 'UTIL_JUL', 'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR', 'UTIL_AVG',
       'UTIL_MAX', 'UTIL_MIN', 'UTIL_STD', 'MONTHS_NO_BALANCE',
       'BILL_GROWTH_RATIO', 'UTIL_TREND', 'PAY_TREND'],
      dtype='str')

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [10]:
logreg = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0, add_indicator=True)),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [12]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

### Using all engineered features

In [13]:
for name, m in [('logreg', logreg), ('xgb', model)]:
    auc = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc')
    ap = cross_val_score(m, X_train, y_train, cv=cv, scoring='average_precision')
    print('%s: AUC %.4f +/- %.4f, PR-AUC %.4f' % (name, auc.mean(), auc.std(), ap.mean()))

logreg: AUC 0.6174 +/- 0.0132, PR-AUC 0.1907


xgb: AUC 0.6383 +/- 0.0094, PR-AUC 0.2247


### Ablation

In [14]:
sets = {
    'raw': raw_cols,
    '+ payment ratios': raw_cols + payment_ratio_cols,
    '+ utilization': raw_cols + payment_ratio_cols + util_rate_cols,
    '+ aggregates': raw_cols + payment_ratio_cols + util_rate_cols + aggregate_cols,
}

for name, m in [('logreg', logreg), ('xgb', model)]:
    for n, cols in sets.items():
        auc = cross_val_score(m, X_train[cols], y_train, cv=cv, scoring='roc_auc')
        print('%s - %s: AUC %.4f +/- %.4f' % (name, n, auc.mean(), auc.std()))

logreg - raw: AUC 0.6149 +/- 0.0161


logreg - + payment ratios: AUC 0.6161 +/- 0.0123


logreg - + utilization: AUC 0.6173 +/- 0.0132


logreg - + aggregates: AUC 0.6174 +/- 0.0132


xgb - raw: AUC 0.6363 +/- 0.0118


xgb - + payment ratios: AUC 0.6324 +/- 0.0111


xgb - + utilization: AUC 0.6379 +/- 0.0111


xgb - + aggregates: AUC 0.6383 +/- 0.0094
